In [1]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

Author: Manuel Elias Orellana Lavayen

Date: 2026-10-05

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.1

debugpy: 1.8.22



## Importando Librerias

In [1]:
import polars as pl
import numpy as np
import os
import json
from pathlib import Path
ruta_base = Path.cwd()
ruta_base

WindowsPath('C:/Users/ManuelOrellana/Desktop/Proyectos Aplicaciones/Aplicacion Prediccion Productos/Creacion de Modelos/Analisis y entrenamiento de modelos')

## Cargando Dataset

In [3]:
df_unidades= pl.read_parquet(ruta_base / "Datasets/Concatenados/Limpios/train.parquet", columns= ["ID", "unidades_vendidas"],  low_memory=False)
df_unidades.head()

ID,unidades_vendidas
u32,f32
91293,13.014
91320,2.0
91357,4.0
91366,3.0
91372,5.0


## Visualizando y convirtiendo valores negativos a 0 en 'unidades_vendidas'

In [4]:
print("====== Valores Negativos======")
print()
print(len(df_unidades.filter(pl.col("unidades_vendidas") < 0)))

====== Valores Negativos======

7795


In [5]:
df_unidades = df_unidades.with_columns(pl.when(pl.col("unidades_vendidas") < 0).then(0).otherwise(pl.col("unidades_vendidas")).alias("unidades_vendidas"))
df_unidades.head()

ID,unidades_vendidas
u32,f32
91293,13.014
91320,2.0
91357,4.0
91366,3.0
91372,5.0


## Guardando Dataset con 'unidades_vendidas' negativas en 0

In [6]:
df_unidades = df_unidades.lazy()

df = (
    pl.scan_parquet(ruta_base/ "Datasets/Concatenados/Limpios/train.parquet")
    .drop("unidades_vendidas")
    .join(df_unidades,
          on = "ID",
          how = "left"
          ))

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

# Creando Nuevas Variables

## año, mes, dia semana y semana en seno y coseno

In [7]:
DOS_PI = 2 * np.pi

diccionario_tipos_columnas = {
    "en_promocion": pl.Int8,
    "es_feriado_nacional": pl.Int8,
    "es_feriado_regional": pl.Int8,
    "es_feriado_local": pl.Int8,
    "es_evento": pl.Int8,
}


df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .with_columns(pl.col("fecha").dt.year().alias("año")).cast({"año": pl.Int16})
    .with_columns(pl.col("fecha").dt.month().alias("mes")).cast({"mes": pl.Int8})
    .with_columns(pl.col("fecha").dt.week().alias("semana")).cast({"semana": pl.Int8})
    .with_columns(pl.col("fecha").dt.weekday().alias("dia_semana")).cast({"dia_semana": pl.Int8})
    .with_columns(pl.col("fecha").dt.day().alias("dia")).cast({"dia": pl.Int8})

    .with_columns(((pl.col("mes") * DOS_PI) / 12).cast(pl.Float32).alias("mes_radianes"))
    .with_columns(((pl.col("dia_semana") * DOS_PI) / 7).cast(pl.Float32).alias("dia_semana_radianes"))
    .with_columns(((pl.col("semana") * DOS_PI) / 53).cast(pl.Float32).alias("semana_radianes"))

    .with_columns((pl.col("mes_radianes").sin().alias("mes_seno"))).cast({"mes_seno": pl.Float32})
    .with_columns((pl.col("mes_radianes").cos().alias("mes_coseno"))).cast({"mes_coseno": pl.Float32})
    .with_columns((pl.col("dia_semana_radianes").sin().alias("dia_semana_seno"))).cast({"dia_semana_seno": pl.Float32})
    .with_columns((pl.col("dia_semana_radianes").cos().alias("dia_semana_coseno"))).cast({"dia_semana_coseno": pl.Float32})
    .with_columns((pl.col("semana_radianes").sin().alias("semana_seno"))).cast({"semana_seno": pl.Float32})
    .with_columns((pl.col("semana_radianes").cos().alias("semana_coseno"))).cast({"semana_coseno": pl.Float32})
    .cast(diccionario_tipos_columnas)
)

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

In [8]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet", ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

## Si cae quincena o no

In [9]:
df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .with_columns(
        pl.when((pl.col("dia") == 15) | (pl.col("dia") == 30) | (pl.col("dia") == 31))
        .then(1)
        .otherwise(0)
        .alias("es_quincena")
    )
    .cast({"es_quincena": pl.Int8})
)

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

In [10]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet", ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

## Si un producto tiene Mas de 100 unidades vendidas

In [11]:
df  = (pl.scan_parquet (ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
       .with_columns(pl.when(pl.col("unidades_vendidas") > 100)
                     .then(1)
                     .otherwise(0)
                     .alias("venta_grande")
                     )
       .cast({"venta_grande": pl.Int8})
       )

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

In [12]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet", ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

## Si un producto se vende o no

In [13]:
df  = (pl.scan_parquet (ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
       .with_columns(pl.when(pl.col("unidades_vendidas") > 0)
                     .then(1)
                     .otherwise(0)
                     .alias("Se vende")
                     )
       .cast({"Se vende": pl.Int8})
       )

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

In [14]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet", ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

## Unidades vendidas el dia anterior sobre ese producto en ese mismo local

In [15]:
df_unidades_log = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .select(["ID", "fecha", "unidades_vendidas", "numero_tienda", "numero_articulo"])
    .sort("numero_tienda", "numero_articulo", "fecha")
    .collect(engine = "streaming"))

df_unidades_log = (df_unidades_log.with_columns(
        pl.col("unidades_vendidas")
        .shift(1)
        .over(["numero_tienda", "numero_articulo"])
        .fill_null(0) # Rellenamos los nulos del primer día con 0
        .alias("unidades_vendidas_ayer")
    )
    .cast({"unidades_vendidas_ayer": pl.Float32}))

df_unidades_log = df_unidades_log.drop(["numero_tienda", "numero_articulo", "fecha", "unidades_vendidas"])
df_unidades_log = df_unidades_log.lazy()

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .join(
        df_unidades_log,
        on = "ID",
        how = "left"
    )
)

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

In [16]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet", ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

## Promedio de Unidades vendidas en los ultimos 7 dias sobre ese producto en ese mismo local

In [17]:
df_promedio_unidades_log = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .select(["ID", "fecha", "unidades_vendidas", "numero_tienda", "numero_articulo"])
    .sort("numero_tienda", "numero_articulo", "fecha")
    .collect(engine = "streaming"))

df_promedio_unidades_log = (
    df_promedio_unidades_log
    .with_columns(
        pl.col("unidades_vendidas")
        .shift(1)
        .rolling_mean(window_size=7, min_samples=1) # Calcula el promedio de los días anteriores, (1 semana máximo y minimo 1 día)
        .over(["numero_tienda", "numero_articulo"])
        .fill_null(0) # Rellenamos los nulos del primer día con 0
        .alias("unidades_vendidas_promedio_semanal")
    )
    .cast({"unidades_vendidas_promedio_semanal": pl.Float32}))

df_promedio_unidades_log = df_promedio_unidades_log.drop(["fecha", "unidades_vendidas", "numero_tienda", "numero_articulo"])
df_promedio_unidades_log = df_promedio_unidades_log.lazy()

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .join(
        df_promedio_unidades_log,
        on= "ID",
        how= "left"
    )
)

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

In [18]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet", ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

### Promedio de petróleo de los ultimos 90 dias

In [2]:
df_promedio_petroleo = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .select(["ID", "fecha", "precio_petroleo_wti", "numero_tienda", "numero_articulo"])
    .sort("numero_tienda", "numero_articulo", "fecha")
    .collect(engine= "streaming"))

df_promedio_petroleo = (
    df_promedio_petroleo
    .with_columns(
            pl.col("precio_petroleo_wti")
            .shift(1)
            .rolling_mean(window_size=90, min_samples=1) # Calcula el promedio de los días anteriores, (3 meses máximo y minimo 1 día)
            .over(["numero_tienda", "numero_articulo"])
            .fill_null(pl.col("precio_petroleo_wti")) # Rellenamos los nulos del primer día con el valor del mismo día
            .alias("petroleo_promedio_3meses")
        )
    .cast({"petroleo_promedio_3meses": pl.Float32}))

df_promedio_petroleo = df_promedio_petroleo.drop(["fecha", "precio_petroleo_wti", "numero_tienda", "numero_articulo"])
df_promedio_petroleo = df_promedio_petroleo.lazy()

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .join(
        df_promedio_petroleo,
        on = "ID",
        how = "left"
    )
)

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

In [3]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet", ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")

# Codificando Variables Categoricas

### clase_producto

In [4]:
clases_unicas = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .select(["clase_producto"])
    .cast({"clase_producto": pl.String})
    .unique()
    .collect(engine= "streaming")
    .get_column("clase_producto")
    .to_list()
)

mapeo_clases_unicas = {}

for numero, texto in enumerate(clases_unicas):
    mapeo_clases_unicas[texto] = numero

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train2.parquet")
    .with_columns(pl.col("clase_producto")
                  .replace(mapeo_clases_unicas))
    .cast({"clase_producto": pl.Int16}))

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")


# Guardando el diccionario de clases unicas como json, para codificar elementos nuevos
with open(f"{ruta_base}/Codificadores Categoricas/clase_producto.json", "w", encoding="utf-8") as archivo:
    json.dump(mapeo_clases_unicas, archivo, indent=4, ensure_ascii=False)

### familia_producto

In [5]:
clases_unicas = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .select(["familia_producto"])
    .cast({"familia_producto": pl.String})
    .unique()
    .collect(engine= "streaming")
    .get_column("familia_producto")
    .to_list()
)

mapeo_clases_unicas = {}

for numero, texto in enumerate(clases_unicas):
    mapeo_clases_unicas[texto] = numero

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .with_columns(pl.col("familia_producto")
                  .cast(pl.String)
                  .replace(mapeo_clases_unicas))
    .cast({"familia_producto": pl.Int16}))

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet")


# Guardando el diccionario de clases unicas como json, para codificar elementos nuevos
with open(f"{ruta_base}/Codificadores Categoricas/familia_producto.json", "w", encoding="utf-8") as archivo:
    json.dump(mapeo_clases_unicas, archivo, indent=4, ensure_ascii=False)

In [6]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet", ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

### ciudad

In [7]:
clases_unicas = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .select(["ciudad"])
    .cast({"ciudad": pl.String})
    .unique()
    .collect(engine= "streaming")
    .get_column("ciudad")
    .to_list()
)

mapeo_clases_unicas = {}

for numero, texto in enumerate(clases_unicas):
    mapeo_clases_unicas[texto] = numero

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .with_columns(pl.col("ciudad")
                  .cast(pl.String)
                  .replace(mapeo_clases_unicas))
    .cast({"ciudad": pl.Int16}))

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet")


# Guardando el diccionario de clases unicas como json, para codificar elementos nuevos
with open(f"{ruta_base}/Codificadores Categoricas/ciudad.json", "w", encoding="utf-8") as archivo:
    json.dump(mapeo_clases_unicas, archivo, indent=4, ensure_ascii=False)

In [8]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet", ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

### provincia_estado

In [9]:
clases_unicas = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .select(["provincia_estado"])
    .cast({"provincia_estado": pl.String})
    .unique()
    .collect(engine= "streaming")
    .get_column("provincia_estado")
    .to_list()
)

mapeo_clases_unicas = {}

for numero, texto in enumerate(clases_unicas):
    mapeo_clases_unicas[texto] = numero

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .with_columns(pl.col("provincia_estado")
                  .cast(pl.String)
                  .replace(mapeo_clases_unicas))
    .cast({"provincia_estado": pl.Int16}))

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet")


# Guardando el diccionario de clases unicas como json, para codificar elementos nuevos
with open(f"{ruta_base}/Codificadores Categoricas/provincia_estado.json", "w", encoding="utf-8") as archivo:
    json.dump(mapeo_clases_unicas, archivo, indent=4, ensure_ascii=False)

In [10]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet", ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

### tipo_tienda

In [11]:
clases_unicas = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .select(["tipo_tienda"])
    .cast({"tipo_tienda": pl.String})
    .unique()
    .collect(engine= "streaming")
    .get_column("tipo_tienda")
    .to_list()
)

mapeo_clases_unicas = {}

for numero, texto in enumerate(clases_unicas):
    mapeo_clases_unicas[texto] = numero

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .with_columns(pl.col("tipo_tienda")
                  .cast(pl.String)
                  .replace(mapeo_clases_unicas))
    .cast({"tipo_tienda": pl.Int16}))

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet")


# Guardando el diccionario de clases unicas como json, para codificar elementos nuevos
with open(f"{ruta_base}/Codificadores Categoricas/tipo_tienda.json", "w", encoding="utf-8") as archivo:
    json.dump(mapeo_clases_unicas, archivo, indent=4, ensure_ascii=False)

In [12]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet", ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")

### numero_articulo

In [13]:
clases_unicas = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .select(["numero_articulo"])
    .cast({"numero_articulo": pl.String})
    .unique()
    .collect(engine= "streaming")
    .get_column("numero_articulo")
    .to_list()
)

mapeo_clases_unicas = {}

for numero, texto in enumerate(clases_unicas):
    mapeo_clases_unicas[texto] = numero

df = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
    .with_columns(pl.col("numero_articulo")
                  .cast(pl.String)
                  .replace(mapeo_clases_unicas))
    .cast({"numero_articulo": pl.Int16}))

df.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet")


# Guardando el diccionario de clases unicas como json, para codificar elementos nuevos
with open(f"{ruta_base}/Codificadores Categoricas/numero_articulo.json", "w", encoding="utf-8") as archivo:
    json.dump(mapeo_clases_unicas, archivo, indent=4, ensure_ascii=False)

In [14]:
os.remove(ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")
os.rename(ruta_base / "Datasets/Concatenados/Limpios/train4.parquet", ruta_base / "Datasets/Concatenados/Limpios/train3.parquet")